In [1]:
import pandas as pd
import re

In [2]:
REGION = 'CMacedonia'
enc = 'utf-8'

In [3]:
data = pd.read_csv(f'../../data/{REGION}/GR_{REGION}_GRID_Landcover_2001-2022.csv', encoding=enc)
data.columns = data.columns.str.lower()

In [4]:
data.head()

,x,y,2001_01_01_lc_prop1,2001_01_01_lc_prop1_assessment,2001_01_01_lc_prop2,2001_01_01_lc_prop2_assessment,2001_01_01_lc_prop3,2001_01_01_lc_prop3_assessment,2001_01_01_lc_type1,2001_01_01_lc_type2,...,2022_01_01_lc_prop2_assessment,2022_01_01_lc_prop3,2022_01_01_lc_prop3_assessment,2022_01_01_lc_type1,2022_01_01_lc_type2,2022_01_01_lc_type3,2022_01_01_lc_type4,2022_01_01_lc_type5,2022_01_01_lw,2022_01_01_qc
0,23.61372,39.93566,22,70.0,20,70.0,20,70.0,9,9,...,78.0,20,78.0,8,8,4,1,1,2,0
1,23.63714,39.93566,31,74.0,30,74.0,30,74.0,10,10,...,79.0,20,79.0,8,8,4,1,1,2,0
2,23.66057,39.93566,21,68.0,20,68.0,20,68.0,8,8,...,77.0,20,77.0,8,8,4,1,1,2,0
3,23.56687,39.95363,21,72.0,20,72.0,20,72.0,8,8,...,77.0,20,77.0,8,8,4,2,2,2,0
4,23.59030,39.95363,21,69.0,20,69.0,20,69.0,8,8,...,88.0,20,88.0,9,9,4,2,2,2,0


In [5]:
lc_columns = data.columns.tolist()
lc_columns = [x.split('_')[0] for x in lc_columns]

lc_columns
lc_years = set()

for i in range(0, len(lc_columns)):
    try:
        lc_years.add(int(lc_columns[i]))
    except ValueError:
        pass

lc_years = list(lc_years)

In [6]:
dates_ = re.compile(r"[0-9]{4}_[0-9]{2}_[0-9]{2}_")
landcover_datasets_per_year = []

for year in lc_years:
    frame_name = f'landcover_{year}'
    location_columns = ['x', 'y']
    filtered_columns = [col for col in data if col.startswith(f'{year}')]
    columns_to_keep = location_columns + filtered_columns
    locals()[frame_name] = data[columns_to_keep].copy()
    locals()[frame_name].insert(2, 'year', int(year))
    locals()[frame_name] = locals()[frame_name].rename(columns=lambda x: re.sub(dates_,'',x))
    landcover_datasets_per_year.append(locals()[frame_name])

In [9]:
landcover_processed = pd.concat(landcover_datasets_per_year, ignore_index=True)
landcover_processed.reset_index(drop=True, inplace=True)

In [10]:
landcover_processed

,x,y,year,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc
0,23.61372,39.93566,2001,22,70.0,20,70.0,20,70.0,9,9,4,1,1,2,0
1,23.63714,39.93566,2001,31,74.0,30,74.0,30,74.0,10,10,1,6,6,2,0
2,23.66057,39.93566,2001,21,68.0,20,68.0,20,68.0,8,8,4,4,4,2,0
3,23.56687,39.95363,2001,21,72.0,20,72.0,20,72.0,8,8,4,1,1,2,0
4,23.59030,39.95363,2001,21,69.0,20,69.0,20,69.0,8,8,4,1,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97147,23.52003,41.37300,2022,14,96.0,10,95.0,10,95.0,4,4,6,4,4,2,0
97148,23.54345,41.37300,2022,31,93.0,30,93.0,30,93.0,10,10,1,6,6,2,0
97149,23.56687,41.37300,2022,22,73.0,20,73.0,20,73.0,9,9,4,1,1,2,0
97150,23.44976,41.39096,2022,31,88.0,30,88.0,30,88.0,10,10,1,6,6,2,0


In [13]:
for year in landcover_processed['year'].sort_values().unique():
    subset = landcover_processed[landcover_processed['year'] == year]
    subset.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_{year}.csv', index=False, encoding=enc)

In [12]:
landcover_processed.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_2001-2024_processed.csv', index=False, encoding=enc)